# 07 - Model Comparison

## Objective

In notebook 06 simple models were almost perfect. One feature, `ratio_is_one`, does most of the work. This is probably a simulation effect, so a comparison on that feature set cannot tell the models apart.

Here we:
1. Test the single rule `ratio_is_one` alone.
2. Remove the ratio features and see what the other features can do (harder feature sets).
3. Compare 4 models (5 if LightGBM is installed) on the easy and on the hard feature set, with a confidence interval.
4. Compare ways to handle the imbalance.

Only the validation set is used. The test set is still untouched.

## Imports and Data Loading

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

sys.path.append('..')
from src.features import config
from src.data.splits import temporal_masks, TRAIN_END_STEP, VAL_END_STEP
from src.models.feature_sets import FEATURE_SETS, HARD_SETS, add_ratio_features, prepare_linear
from src.models.evaluation import evaluate, bootstrap_pr_auc

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
SEED = config.RANDOM_SEED

In [2]:
table = add_ratio_features(pd.read_parquet(config.PROCESSED_PATH))
in_scope = (table['type_CASH_OUT'] + table['type_TRANSFER']) == 1
data = table[in_scope].reset_index(drop=True)
del table

tr_mask, va_mask, _ = temporal_masks(data, TRAIN_END_STEP, VAL_END_STEP)
tr, va = data[tr_mask], data[va_mask]
ytr, yva = tr['isFraud'].to_numpy(), va['isFraud'].to_numpy()
print(f'train {len(tr):,} rows ({ytr.sum():,} frauds), validation {len(va):,} rows ({yva.sum():,} frauds)')

train 2,645,779 rows (5,561 frauds), validation 86,155 rows (1,286 frauds)


## 1. The single rule: ratio_is_one

If `ratio_is_one` = 1 (amount equals the origin balance) we say fraud, otherwise legit. No model.

In [3]:
def rule_report(df, name):
    flag = df['ratio_is_one'] == 1
    y = df['isFraud'] == 1
    tp, fp, fn = int((flag & y).sum()), int((flag & ~y).sum()), int((~flag & y).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return {'set': name, 'rows': len(df), 'frauds': int(y.sum()), 'flagged': tp + fp,
            'caught': tp, 'false_alerts': fp, 'missed': fn,
            'precision': round(precision, 4), 'recall': round(recall, 4),
            'f1': round(2 * precision * recall / max(precision + recall, 1e-12), 4)}

rule_table = pd.DataFrame([rule_report(tr, 'train'), rule_report(va, 'validation')])
rule_table

,set,rows,frauds,flagged,caught,false_alerts,missed,precision,recall,f1
0,train,2645779,5561,5440,5440,0,121,1.0,0.9782,0.9890
1,validation,86155,1286,1264,1264,0,22,1.0,0.9829,0.9914


In [4]:
# Who are the false alerts and the misses of the rule? (validation)
flag = va['ratio_is_one'] == 1
fp_rows = va[flag & (va['isFraud'] == 0)]
fn_rows = va[~flag & (va['isFraud'] == 1)]
print('False alerts by type (1 = TRANSFER):')
print(fp_rows['type_TRANSFER'].value_counts().to_string())
print()
print('Missed frauds: zero origin balance =', int(fn_rows['zero_old_balance_orig'].sum()), 'of', len(fn_rows))
print('Missed frauds: destination is new  =', int(fn_rows['dest_is_new'].sum()), 'of', len(fn_rows))

False alerts by type (1 = TRANSFER):
Series([], )

Missed frauds: zero origin balance = 3 of 22
Missed frauds: destination is new  = 20 of 22


**Observation:** The rule alone has precision 1.0 and recall 0.983 on validation (0 false alerts, 22 frauds missed). On train it is the same: 0 false alerts in 2.6M rows. Real data would never be so clean. So "amount = balance" is a simulation effect: fraud takes the full balance and normal transactions never do.

Of the 22 missed frauds, 3 have a zero origin balance and 20 go to a new destination. This is what a model can still add on top of the rule.

## 2. Harder feature sets

We remove the ratio features. `amount_log` and `oldbalanceOrg_log` stay in `H1`. A model could still find "amount = balance" from these two. So we also test `H2`, which has no origin balance at all.

- `E1_core`: easy set (the S1 set of notebook 06)
- `H1_no_ratio`: no ratio features, with destination history
- `H1b_no_ratio_no_dest`: no ratio features, no destination history
- `H2_no_origin_balance`: no ratio features, no origin balance, with destination history

In [5]:
SETS = {'E1_core': FEATURE_SETS['S1_core'], **HARD_SETS}
for name, cols in SETS.items():
    print(f'{name} ({len(cols)}): {cols}')

E1_core (8): ['amount_log', 'oldbalanceOrg_log', 'oldbalanceDest_log', 'amount_to_balance_ratio', 'zero_old_balance_orig', 'ratio_gap_to_one', 'ratio_is_one', 'type_TRANSFER']
H1_no_ratio (11): ['amount_log', 'oldbalanceOrg_log', 'oldbalanceDest_log', 'zero_old_balance_orig', 'type_TRANSFER', 'dest_txn_count_before', 'dest_avg_amount_before', 'dest_is_new', 'dest_txn_count_recent', 'dest_amount_sum_recent', 'dest_steps_since_last']
H1b_no_ratio_no_dest (5): ['amount_log', 'oldbalanceOrg_log', 'oldbalanceDest_log', 'zero_old_balance_orig', 'type_TRANSFER']
H2_no_origin_balance (9): ['amount_log', 'oldbalanceDest_log', 'type_TRANSFER', 'dest_txn_count_before', 'dest_avg_amount_before', 'dest_is_new', 'dest_txn_count_recent', 'dest_amount_sum_recent', 'dest_steps_since_last']


## 3. Helper

In [6]:
results = []

def run(model, name, set_name, cols, linear=False, train=None, note=''):
    Xtr, y_fit = train if train is not None else (tr[cols], ytr)
    Xva = va[cols]
    if linear:
        Xtr, Xva = prepare_linear(Xtr), prepare_linear(Xva)
    start = time.time()
    model.fit(Xtr, y_fit)
    secs = time.time() - start
    scores = model.predict_proba(Xva)[:, 1]
    row = {'model': name, 'features': set_name, 'note': note, 'seconds': round(secs, 1)}
    row.update(evaluate(yva, scores))
    row['pr_auc_low'], row['pr_auc_high'] = bootstrap_pr_auc(yva, scores)
    results.append(row)
    return scores

def make_models():
    models = {
        'Logistic Regression': (make_pipeline(StandardScaler(),
                                LogisticRegression(max_iter=300, class_weight='balanced')), True),
        'Decision Tree (depth 6)': (DecisionTreeClassifier(max_depth=6, min_samples_leaf=100,
                                    random_state=SEED), False),
        'Random Forest': (RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=20,
                          max_samples=0.2, n_jobs=-1, random_state=SEED), False),
        'Gradient Boosting': (HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,
                              random_state=SEED), False),
    }
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = (LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                              random_state=SEED, verbose=-1), False)
    except ImportError:
        print('lightgbm is not installed, skipped')
    return models

def show(df):
    cols = ['model', 'features', 'note', 'pr_auc', 'pr_auc_low', 'pr_auc_high', 'roc_auc',
            'best_f1', 'recall_at_p80', 'recall_at_p95', 'alerts_per_10k_at_p95', 'seconds']
    return df[cols].round(4)

## 4. Model comparison: easy set vs hard set

PR-AUC comes with a 95% interval (resampling the validation rows 100 times). If two intervals overlap a lot, we cannot say one model is better.

In [7]:
scores_store = {}
for set_name in ['E1_core', 'H1_no_ratio']:
    for model_name, (model, linear) in make_models().items():
        scores_store[(model_name, set_name)] = run(model, model_name, set_name, SETS[set_name], linear=linear)

comparison = pd.DataFrame(results)
show(comparison.sort_values(['features', 'pr_auc'], ascending=[True, False]))

,model,features,note,pr_auc,pr_auc_low,pr_auc_high,roc_auc,best_f1,recall_at_p80,recall_at_p95,alerts_per_10k_at_p95,seconds
2,Random Forest,E1_core,,1.0000,1.0000,1.0000,1.0000,0.9996,1.0000,1.0000,156.9265,32.6
1,Decision Tree (depth 6),E1_core,,1.0000,1.0000,1.0000,1.0000,0.9988,1.0000,1.0000,149.8462,19.9
3,Gradient Boosting,E1_core,,0.9999,0.9997,1.0000,1.0000,0.9988,1.0000,0.9984,151.8194,4.2
0,Logistic Regression,E1_core,,0.9998,0.9995,1.0000,1.0000,0.9992,1.0000,0.9984,156.8104,5.2
4,LightGBM,E1_core,,0.4627,0.4296,0.4908,0.7253,0.6571,0.5358,0.0000,0.0000,21.1
7,Random Forest,H1_no_ratio,,0.7971,0.7776,0.8146,0.9762,0.8082,0.7488,0.6757,106.0879,47.0
9,LightGBM,H1_no_ratio,,0.7212,0.6979,0.7449,0.8991,0.7228,0.6384,0.0000,0.0000,28.7
8,Gradient Boosting,H1_no_ratio,,0.6729,0.6525,0.7014,0.8816,0.7458,0.6446,0.0000,0.0000,10.5
6,Decision Tree (depth 6),H1_no_ratio,,0.6664,0.6449,0.6917,0.9629,0.6672,0.4907,0.3204,47.8208,24.9
5,Logistic Regression,H1_no_ratio,,0.1519,0.1421,0.1643,0.9422,0.2630,0.0000,0.0000,0.0000,8.9


**Observation:**
- Easy set: Random Forest, Decision Tree, Gradient Boosting and Logistic Regression all get a PR-AUC between 0.9998 and 1.0000. We cannot tell them apart.
- LightGBM gets only 0.46 on the easy set. This is wrong for such an easy task, so something is off in this run. We check it in the next notebook.
- Hard set (no ratio): Random Forest is best (0.797), then LightGBM (0.721), Gradient Boosting (0.673) and the tree (0.666). Logistic Regression is very weak (0.152) because the signal is not linear.
- The Random Forest interval does not overlap with LightGBM, so it is really better here.

## 5. What matters on the hard set?

Gradient Boosting on three hard sets. This shows what the destination history and the origin balance add.

In [8]:
start_n = len(results)
for set_name in ['H1_no_ratio', 'H1b_no_ratio_no_dest', 'H2_no_origin_balance']:
    model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=SEED)
    run(model, 'Gradient Boosting', set_name, SETS[set_name], note='ablation')

ablation = pd.DataFrame(results[start_n:])
show(ablation)

,model,features,note,pr_auc,pr_auc_low,pr_auc_high,roc_auc,best_f1,recall_at_p80,recall_at_p95,alerts_per_10k_at_p95,seconds
0,Gradient Boosting,H1_no_ratio,ablation,0.6729,0.6525,0.7014,0.8816,0.7458,0.6446,0.0000,0.0000,11.1
1,Gradient Boosting,H1b_no_ratio_no_dest,ablation,0.5076,0.4785,0.5330,0.6835,0.6140,0.4712,0.0000,0.0000,8.3
2,Gradient Boosting,H2_no_origin_balance,ablation,0.4180,0.3908,0.4453,0.8761,0.4691,0.1656,0.1633,25.6514,10.7


**Observation:** When the ratio is removed, destination history helps a lot: PR-AUC goes from 0.508 (no destination features) to 0.673. We keep these features. The origin balance also matters: without it the PR-AUC falls to 0.418 (H2). But H2 still has a ROC-AUC of 0.876, so the other features carry some signal.

Note: H1 still has `amount_log` and `oldbalanceOrg_log`. A model can use both to find "amount = balance" again. H2 is the cleaner test.

## 6. Handling the imbalance

On `H1_no_ratio`, with Gradient Boosting:
- no balancing
- class weights
- undersampling: keep all frauds and 20 legit rows for each fraud
- undersampling + SMOTE (needs `imbalanced-learn`)

Balancing is done on the training data only. SMOTE makes new rows by mixing two frauds, so 0/1 columns like `dest_is_new` can get values between 0 and 1.

In [9]:
cols = SETS['H1_no_ratio']
start_n = len(results)

def hgb(**kw):
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=SEED, **kw)

run(hgb(), 'Gradient Boosting', 'H1_no_ratio', cols, note='no balancing')
run(hgb(class_weight='balanced'), 'Gradient Boosting', 'H1_no_ratio', cols, note='class weights')

rng = np.random.default_rng(SEED)
pos = np.flatnonzero(ytr == 1)
neg = np.flatnonzero(ytr == 0)
keep = np.sort(np.concatenate([pos, rng.choice(neg, size=len(pos) * 20, replace=False)]))
X_us, y_us = tr[cols].iloc[keep], ytr[keep]
run(hgb(), 'Gradient Boosting', 'H1_no_ratio', cols, train=(X_us, y_us), note='undersampling 20:1')

try:
    from imblearn.over_sampling import SMOTE
    X_sm, y_sm = SMOTE(sampling_strategy=0.5, random_state=SEED).fit_resample(X_us, y_us)
    run(hgb(), 'Gradient Boosting', 'H1_no_ratio', cols, train=(X_sm, y_sm), note='undersampling + SMOTE')
except ImportError:
    print('imbalanced-learn is not installed, SMOTE skipped')

balance = pd.DataFrame(results[start_n:])
show(balance)

,model,features,note,pr_auc,pr_auc_low,pr_auc_high,roc_auc,best_f1,recall_at_p80,recall_at_p95,alerts_per_10k_at_p95,seconds
0,Gradient Boosting,H1_no_ratio,no balancing,0.6729,0.6525,0.7014,0.8816,0.7458,0.6446,0.0000,0.0000,11.1
1,Gradient Boosting,H1_no_ratio,class weights,0.9305,0.9223,0.9403,0.9984,0.8556,0.8577,0.7613,119.5520,55.2
2,Gradient Boosting,H1_no_ratio,undersampling 20:1,0.9075,0.8968,0.9184,0.9975,0.8358,0.8289,0.7348,115.3735,0.9
3,Gradient Boosting,H1_no_ratio,undersampling + SMOTE,0.9303,0.9226,0.9403,0.9982,0.8509,0.8569,0.7667,120.3645,2.8


**Observation:** Balancing changes a lot. Gradient Boosting on H1 goes from 0.673 (no balancing) to 0.931 with class weights, 0.908 with undersampling and 0.930 with SMOTE. Class weights and SMOTE are the same inside the interval. Undersampling is a bit lower.

A jump to 0.93 is big. With class weights the model probably finds "amount = balance" again from the two log columns. So H1 is not really a hard set, and 0.93 does not mean the other features are strong.

Class weights are the simplest and work as well as SMOTE, so we use them. Also, the comparison in section 4 was not fair, because only Logistic Regression had class weights. We redo it in notebook 08.

## 7. Conclusion and next step

**Findings**
- One rule ("amount = balance") gets 100% precision and 98.3% recall. This is a PaySim effect. Real fraud would not be so clean.
- On the easy set all models are the same (PR-AUC about 1.0).
- Without the ratio features the problem is harder. Destination history helps (0.51 to 0.67).
- Class weights matter a lot (0.67 to 0.93).
- Only Logistic Regression was balanced in section 4, so that comparison is not fair. LightGBM also gave a strange result on the easy set (0.46).

**Decisions**
- Use class weights for all models from now on. Drop SMOTE and undersampling (no clear gain, more work).
- Keep the destination history features.
- Do not choose the final model from the easy set.

**Next notebook:** `08_balanced_comparison.ipynb`. Same models, all with class weights, on E1, H1 and H2. It also checks LightGBM, and checks if the best model catches frauds that the rule misses.